# Module 02 — Gemini + Groq APIs to LangChain (Google Colab)

This notebook is for **Google Colab**.

You will:
1. Install everything first.
2. Load `GEMINI_API_KEY` and `GROQ_API_KEY` from **Colab Secrets**.
3. Call Gemini directly.
4. Call Groq directly.
5. Compare the different API styles.
6. Use LangChain with both providers.
7. Practice normal and fixed prompts.
8. End with a prompt-engineering question.

> Groq is the inference/API platform. It is different from Grok by xAI.


## Before running

In Colab, click the **Secrets / key icon** on the left and add:

```text
GEMINI_API_KEY
GROQ_API_KEY
```

Turn **Notebook access** ON for both. Never paste keys directly into notebook code.


In [ ]:
# CELL 1 — INSTALL EVERYTHING FIRST

!pip -q install -U google-genai groq langchain langchain-core langchain-groq langchain-google-genai

print("All required libraries installed.")


In [ ]:
# CELL 2 — LOAD API KEYS FROM COLAB SECRETS

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

assert GEMINI_API_KEY, "Add GEMINI_API_KEY in Colab Secrets."
assert GROQ_API_KEY, "Add GROQ_API_KEY in Colab Secrets."

print("Gemini key loaded:", bool(GEMINI_API_KEY))
print("Groq key loaded:", bool(GROQ_API_KEY))


# Part 1 — Gemini Direct API

Flow:

```text
Python → Google SDK → Gemini API → Gemini model → answer
```


In [ ]:
# CELL 3 — CREATE GEMINI CLIENT AND ASK A QUESTION

from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

question = "What is artificial intelligence? Explain in 4 simple lines."

gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=question,
)

print(gemini_response.text)


### What happened?

- `genai.Client(...)` creates the Gemini client.
- `generate_content(...)` sends the prompt.
- `contents=` contains our prompt.
- `response.text` gives us the generated text.


# Part 2 — Groq Direct API

Now we do the same conceptual job through Groq:

```text
Python → Groq SDK → Groq API → model → answer
```


In [ ]:
# CELL 4 — CREATE GROQ CLIENT AND ASK A QUESTION

from groq import Groq

GROQ_MODEL = "openai/gpt-oss-20b"

groq_client = Groq(api_key=GROQ_API_KEY)

question = "What is artificial intelligence? Explain in 4 simple lines."

groq_response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "user", "content": question}
    ],
)

print(groq_response.choices[0].message.content)


# Compare the direct APIs

Gemini:

```python
response = gemini_client.models.generate_content(...)
print(response.text)
```

Groq:

```python
response = groq_client.chat.completions.create(...)
print(response.choices[0].message.content)
```

Same goal, but different clients, request formats, and response formats.

If an application supports many providers, this provider-specific code grows. This leads us to **LangChain**.


# Part 3 — LangChain

LangChain does not replace Gemini or Groq. It gives our application a more consistent way to work with different model providers.

```text
                 ┌─ Gemini
Python → LangChain
                 └─ Groq
```

Think of it like a **universal adapter**.


In [ ]:
# CELL 5 — CREATE LANGCHAIN MODEL OBJECTS

import os

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

gemini_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL)
groq_llm = ChatGroq(model=GROQ_MODEL)

print("Both LangChain model objects are ready.")


## `invoke()` and `.content`

```python
response = llm.invoke(question)
```

For this class, `invoke()` means:

> Send this input through the LangChain model/component and return one result.

A chat model returns a message object such as an `AIMessage`.

```python
response.content
```

contains the model output. Some providers may return plain text while others may return richer content blocks.


In [ ]:
# CELL 6 — GROQ THROUGH LANGCHAIN

question = "What is machine learning? Explain in 4 simple lines."

response = groq_llm.invoke(question)

print(response.content)


In [ ]:
# CELL 7 — GEMINI THROUGH LANGCHAIN

question = "What is machine learning? Explain in 4 simple lines."

response = gemini_llm.invoke(question)

if isinstance(response.content, str):
    print(response.content)
else:
    print(response.content[0]["text"])


# Part 4 — One common function

Now both providers can be stored behind one simple interface.


In [ ]:
# CELL 8 — COMMON FUNCTION

models = {
    "gemini": gemini_llm,
    "groq": groq_llm,
}

def ask_llm(provider, question):
    llm = models[provider]
    response = llm.invoke(question)
    return response.content

print("ask_llm() is ready.")


In [ ]:
# CELL 9 — ASK GROQ

answer = ask_llm(
    provider="groq",
    question="Explain tokens in generative AI in 4 simple lines.",
)

print(answer)


In [ ]:
# CELL 10 — ASK GEMINI

answer = ask_llm(
    provider="gemini",
    question="Explain tokens in generative AI in 4 simple lines.",
)

if isinstance(answer, str):
    print(answer)
else:
    print(answer[0]["text"])


# Part 5 — Normal prompt practice

Change the provider and question without changing the architecture.


In [ ]:
# CELL 11 — INTERACTIVE PRACTICE

provider = input("Choose provider (gemini/groq): ").strip().lower()
question = input("Ask your question: ")

if provider not in models:
    print("Choose either gemini or groq.")
else:
    answer = ask_llm(provider, question)

    print("\n--- ANSWER ---\n")

    if isinstance(answer, str):
        print(answer)
    else:
        print(answer[0]["text"])


# Part 6 — Fixed / Static Prompt

Applications often contain developer-written instructions.

Notice how adding role, format, constraints, and examples changes the response.


In [ ]:
# CELL 12 — STATIC PROMPT

STATIC_PROMPT = '''
You are a beginner-friendly AI teacher.

Explain what an API is.

Rules:
- Use a clear heading.
- Use simple English.
- Use bullet points.
- Give one restaurant waiter analogy.
- Give one software example.
- End with a one-line summary.
'''

answer = ask_llm("gemini", STATIC_PROMPT)

if isinstance(answer, str):
    print(answer)
else:
    print(answer[0]["text"])


# Part 7 — First Prompt Template

A prompt template lets us reuse the same instruction structure while changing variables such as `{topic}`.


In [ ]:
# CELL 13 — LANGCHAIN PROMPT TEMPLATE

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a patient teacher for students with no prior AI knowledge."
    ),
    (
        "human",
        "Explain {topic}. Use simple English, one analogy, and one example."
    ),
])

groq_chain = prompt_template | groq_llm

response = groq_chain.invoke({"topic": "embeddings"})

print(response.content)


# Final Experiment — Bridge to Prompt Engineering

Compare these prompts using the **same model**.

### Prompt A

```text
Explain APIs.
```

### Prompt B

```text
You are teaching a 15-year-old student who has never programmed before.
Explain what an API is in less than 150 words.
Start with a one-sentence definition.
Use a restaurant waiter analogy.
Give one software example.
Finish with one question to test understanding.
```

If the model stays the same but the answer changes because the instructions changed:

> **What exactly are we controlling with the prompt?**

That question leads directly into the next class: **Prompt Engineering**.


In [ ]:
# CELL 14 — COMPARE TWO PROMPTS

prompt_a = "Explain APIs."

prompt_b = '''
You are teaching a 15-year-old student who has never programmed before.
Explain what an API is in less than 150 words.
Start with a one-sentence definition.
Use a restaurant waiter analogy.
Give one software example.
Finish with one question to test understanding.
'''

print("=== PROMPT A RESPONSE ===")
print(groq_llm.invoke(prompt_a).content)

print("\n=== PROMPT B RESPONSE ===")
print(groq_llm.invoke(prompt_b).content)


# Quick Revision

```text
Direct Gemini → response.text
Direct Groq   → response.choices[0].message.content

LangChain:
question → llm.invoke(question) → response.content

Then:
prompt template → model → response

Next class:
Prompt Engineering
```
